# YOLOv11 Detection Training — Chunked Training Pipeline

This notebook trains a **YOLOv11** object detection model to detect individual bees in hive-entrance video frames. It is part of a two-stage bee behavior recognition pipeline.

## Why chunked training?

A single epoch takes **10–13 minutes** on the available hardware, making it impractical to run the full 500-epoch schedule in one session. Instead, training is broken into **chunks** (default: 15 epochs each). Each chunk can be interrupted and resumed safely because:

- **`chunk_state.yaml`** tracks the true global epoch number (Ultralytics resets this on resume)
- **`early_stop_state.yaml`** tracks patience manually across chunks (Ultralytics' built-in early stopping resets on resume)
- **`results_master.csv`** merges per-chunk CSVs into one continuous training log

## How to use

1. Set your config paths in **Cell 2**
2. Run all cells — the notebook auto-detects whether to start fresh or resume
3. Re-run from **Cell 5** each new session to continue training

In [ ]:
from ultralytics import YOLO
import yaml
import pandas as pd
import torch
import os
import gc
import shutil

# Limit GPU memory so the desktop remains usable during training
torch.cuda.set_per_process_memory_fraction(0.90, device=0)

# Clear any stale allocations from a previous run in the same kernel session
gc.collect()
torch.cuda.empty_cache()

## Configuration

All run-specific settings live here. Changing any of these three paths produces a **new, independent run** — nothing from a previous run is touched.

| Variable | Purpose |
|---|---|
| `augmentation_config` | YAML defining data augmentation parameters (flips, mosaic, etc.) |
| `hyperparams_config` | YAML defining optimizer, LR schedule, weight decay, etc. |
| `base_model_path` | Pretrained YOLO weights to start from (n / s / m variants) |

The `run_name` is derived automatically from these three, so the `runs/detect/` folder always reflects exactly which combination was used.

In [ ]:
# ──── Augmentation & Hyperparameter & Name Configs ────
augmentation_config = 'YAMLs/Augmentation/medium_augmentation.yaml'
hyperparams_config  = 'YAMLs/Hyperparameter/spec_AdamW.yaml'
base_model_path     = 'BaseModels/yolo11m.pt'

model_name = os.path.basename(base_model_path).split('.')[0]
run_name   = (
    f"{model_name}"
    f"-{augmentation_config.split('/')[-1].split('_')[0]}"
    f"-{hyperparams_config.split('/')[-1].split('.')[0]}"
)

# Derived paths used throughout the notebook
checkpoint_path = f"runs/detect/{run_name}/weights/last.pt"
weights_dir     = os.path.dirname(checkpoint_path)

print(f"{'='*60}")
print(f"   Run Name: {run_name}")
print(f"{'='*60}\n")

## Load YAML Configs

Both YAML files are merged into a single `training_config` dict that is unpacked directly into `model.train()`. Ultralytics accepts every augmentation and hyperparameter flag as keyword arguments, so no manual mapping is needed.

In [ ]:
# ──── Load Configs ────
def load_yaml_config(filepath):
    with open(filepath, 'r') as f:
        return yaml.safe_load(f)

augmentation_config_dict = load_yaml_config(augmentation_config)
hyperparams_config_dict  = load_yaml_config(hyperparams_config)

# Merge into one dict — passed as **kwargs to model.train()
training_config = {
    **augmentation_config_dict,
    **hyperparams_config_dict
}

print(f"   Configuration loaded:")
print(f"   ├─ Augmentation:   {augmentation_config.split('/')[-1]}")
print(f"   └─ Hyperparameters: {hyperparams_config.split('/')[-1]}\n")

## Chunk & Epoch Settings

### Why manual patience tracking?

Ultralytics resets its internal early-stopping counter every time training is resumed. If we trained in 15-epoch chunks, the counter would never accumulate past 15, making the built-in patience useless. The solution is:

- Set `patience=0` in `model.train()` to **disable** Ultralytics' early stopping
- Track fitness improvements ourselves across chunks in `early_stop_state.yaml`
- Stop training at the top of the next chunk if patience is exhausted

### Why a `chunk_state.yaml`?

When Ultralytics finishes a chunk it strips the epoch number from `last.pt` (sets it to `-1`). Without an external record we would have no reliable way to know which global epoch we are on when resuming.

In [ ]:
# ──── Chunk & Epoch ────
EPOCHS_PER_CHUNK = 15
TOTAL_EPOCHS     = 500
PATIENCE         = 50

# State files — created inside the run directory so they travel with the run
EARLY_STOP_TRACKER = f"runs/detect/{run_name}/early_stop_state.yaml"
CHUNK_STATE_FILE   = f"runs/detect/{run_name}/chunk_state.yaml"

print(f"{'='*60}")
print(f"   TRAINING IN CHUNKS")
print(f"   ├─ Epochs per chunk: {EPOCHS_PER_CHUNK}")
print(f"   ├─ Total target:     {TOTAL_EPOCHS}")
print(f"   └─ Patience:         {PATIENCE}")
print(f"{'='*60}\n")

## Helper Functions

Five utility functions that handle the bookkeeping between chunks:

| Function | Role |
|---|---|
| `get_checkpoint_epoch` | Reads the epoch stored inside a `.pt` file |
| `save/load_early_stop_state` | Persists patience counter + best fitness across sessions |
| `save/load_chunk_state` | Persists the last completed global epoch |
| `fix_chunk_artifacts` | Renames per-chunk epoch checkpoints to global numbering and merges the results CSV |

In [ ]:
# ──── Helper Functions ────

def get_checkpoint_epoch(path):
    checkpoint = torch.load(path, weights_only=False, map_location='cpu')
    return checkpoint.get('epoch', -1)


def save_early_stop_state(epochs_no_improve, best_fitness):
    os.makedirs(os.path.dirname(EARLY_STOP_TRACKER), exist_ok=True)
    with open(EARLY_STOP_TRACKER, 'w') as f:
        yaml.dump({'epochs_no_improve': epochs_no_improve, 'best_fitness': best_fitness}, f)


def load_early_stop_state():
    if os.path.exists(EARLY_STOP_TRACKER):
        data = yaml.safe_load(open(EARLY_STOP_TRACKER))
        return data.get('epochs_no_improve', 0), data.get('best_fitness', -1.0)
    return 0, -1.0


def save_chunk_state(last_completed_epoch):
    os.makedirs(os.path.dirname(CHUNK_STATE_FILE), exist_ok=True)
    with open(CHUNK_STATE_FILE, 'w') as f:
        yaml.dump({'last_completed_epoch': last_completed_epoch}, f)


def load_chunk_state():
    if os.path.exists(CHUNK_STATE_FILE):
        with open(CHUNK_STATE_FILE) as f:
            return yaml.safe_load(f).get('last_completed_epoch', -1)
    return -1


def fix_chunk_artifacts(chunk_start_epoch, chunk_end_epoch):
    """
    After each chunk:
    1. Append this chunk's results.csv rows (with corrected epoch numbers) to results_master.csv
    2. Rename per-chunk epoch*.pt files to reflect global epoch numbers
    """
    chunk_size  = chunk_end_epoch - chunk_start_epoch
    results_csv = f"runs/detect/{run_name}/results.csv"
    master_csv  = f"runs/detect/{run_name}/results_master.csv"

    # ──── CSV merge ────
    if os.path.exists(results_csv):
        new_df = pd.read_csv(results_csv)
        new_df.columns = new_df.columns.str.strip()  # YOLO pads column names with spaces
        if 'epoch' in new_df.columns:
            new_df['epoch'] = new_df['epoch'] + chunk_start_epoch

        if os.path.exists(master_csv):
            master_df = pd.read_csv(master_csv)
            master_df.columns = master_df.columns.str.strip()
            combined = pd.concat([master_df, new_df], ignore_index=True)
        else:
            combined = new_df

        combined.to_csv(master_csv, index=False)
        combined.to_csv(results_csv, index=False)
        print(f"   CSV: +{len(new_df)} rows → {len(combined)} total epochs logged")

    # ──── Checkpoint rename ────
    # Local names (epoch1.pt … epoch15.pt) → global names (epoch16.pt … epoch30.pt)
    renamed, skipped = [], []
    if os.path.exists(weights_dir):
        for fname in sorted(os.listdir(weights_dir)):
            if fname.startswith('epoch') and fname.endswith('.pt'):
                try:
                    local_ep = int(fname.replace('epoch', '').replace('.pt', ''))
                except ValueError:
                    continue
                if local_ep <= chunk_size:
                    src       = os.path.join(weights_dir, fname)
                    global_ep = chunk_start_epoch + local_ep
                    new_name  = f'epoch{global_ep}.pt'
                    dst       = os.path.join(weights_dir, new_name)
                    if fname != new_name:
                        if not os.path.exists(dst):
                            shutil.copy2(src, dst)
                            renamed.append(f'{fname} → {new_name}')
                        else:
                            skipped.append(new_name)
                        os.remove(src)
    if renamed:
        print(f"   Checkpoints renamed: {renamed}")
    if skipped:
        print(f"   Checkpoints skipped (already exist): {skipped}")

## Resume Logic

Before each chunk the notebook determines **where to start** by checking for state files and checkpoints in priority order:

```
1. chunk_state.yaml exists  → use last.pt, resume from stored epoch       (normal path)
2. No state, last.pt has valid epoch  → bootstrap from it                 (migration path)
3. No state, last.pt has epoch=-1  → fall back to highest epochN.pt       (fallback)
4. Nothing found  → fresh start from base model
```

Early-stopping is checked first — if the patience budget is already exhausted from a previous session, a `SystemExit` is raised immediately rather than wasting time loading the model.

In [ ]:
# ──── Early-stop guard ────
epochs_no_improve, best_fitness = load_early_stop_state()

if epochs_no_improve >= PATIENCE:
    print(f"{'='*60}")
    print(f"   TRAINING STOPPED — Early stopping already triggered.")
    print(f"   ({epochs_no_improve} epochs with no improvement >= patience {PATIENCE})")
    print(f"   Best model: runs/detect/{run_name}/weights/best.pt")
    print(f"{'='*60}\n")
    raise SystemExit("Early stopping already triggered. Training is complete.")

# ──── Determine starting point ────
last_epoch = load_chunk_state()

if last_epoch >= 0:  # Normal path — chunk_state.yaml exists
    current_epoch = last_epoch + 1
    if os.path.exists(checkpoint_path):
        load_path = checkpoint_path
        print(f"   State file: last completed epoch = {last_epoch}")
        print(f"   Resuming from last.pt → starting epoch {current_epoch}")
    else:
        print(f"   State file found (epoch {last_epoch}) but last.pt is missing — searching")
        load_path = None
        if os.path.exists(weights_dir):
            periodic = []
            for f in os.listdir(weights_dir):
                if f.startswith('epoch') and f.endswith('.pt'):
                    try:
                        periodic.append((int(f.replace('epoch','').replace('.pt','')), f))
                    except ValueError:
                        continue
            if periodic:
                _, best_f = max(periodic, key=lambda x: x[0])
                load_path = os.path.join(weights_dir, best_f)
                print(f"   └─ Using: {best_f}")
            else:
                print(f"   └─ No checkpoints found. Starting fresh from base model.")

elif os.path.exists(checkpoint_path):  # Migration path — no state file but last.pt exists
    saved_epoch = get_checkpoint_epoch(checkpoint_path)
    if saved_epoch >= 0:
        current_epoch = saved_epoch + 1
        load_path = checkpoint_path
        print(f"   No state file. Read epoch {saved_epoch} from last.pt.")
        print(f"   Resuming → starting epoch {current_epoch}")
    else:
        print(f"   last.pt has epoch=-1 (stripped) and no state file found.")
        print(f"   Searching for highest periodic checkpoint...")
        periodic = []
        for f in os.listdir(weights_dir):
            if f.startswith('epoch') and f.endswith('.pt'):
                try:
                    periodic.append((int(f.replace('epoch','').replace('.pt','')), f))
                except ValueError:
                    continue
        periodic.sort(reverse=True)
        print(f"   Found: {[f for _, f in periodic[:3]]}")
        if periodic:
            best_n, best_f = periodic[0]
            load_path  = os.path.join(weights_dir, best_f)
            saved_epoch = get_checkpoint_epoch(load_path)
            current_epoch = saved_epoch + 1
            print(f"   └─ Using: {best_f} (epoch {saved_epoch})")
        else:
            print(f"   └─ No periodic checkpoints found. Starting fresh.")
            current_epoch = 0
            load_path = None

else:  # Fresh start
    current_epoch = 0
    load_path = None

# ──── Load model ────
next_target = min(current_epoch + EPOCHS_PER_CHUNK, TOTAL_EPOCHS)

if load_path:
    model = YOLO(load_path)
    print(f"   Loaded: {os.path.basename(load_path)}")
else:
    model = YOLO(base_model_path)
    print(f"   Starting fresh from: {base_model_path}")

print(f"   Training epochs {current_epoch} → {next_target} (+{next_target - current_epoch} epochs)")
print(f"{'='*60}\n")

## Training

### Key training decisions

| Parameter | Value | Reason |
|---|---|---|
| `imgsz` | 640 | Matches the tile size produced by `tiling.py` |
| `batch` | 6 | Hardware limit; gradient accumulation makes it behave like a larger batch |
| `cache` | `'disk'` | RAM cache causes incorrect loss calculations; SSD is fast enough |
| `workers` | 4 | More than 6 causes dataloader bottlenecks on this machine |
| `resume` | `False` | Disabled — resuming via Ultralytics corrupts epoch numbering; handled manually above |
| `patience` | 0 | Ultralytics early stopping disabled; we track it ourselves across chunks |
| `save_period` | 5 | Periodic checkpoints every 5 epochs for fallback recovery |

### Post-chunk bookkeeping

After `model.train()` returns, three things happen:
1. `save_chunk_state` — records the true last global epoch
2. `fix_chunk_artifacts` — renames checkpoints + merges CSVs
3. Early-stop state update — increments patience counter or resets it if fitness improved

In [ ]:
try:
    results = model.train(
        data='YAMLS/dataset.yaml',
        name=run_name,
        resume=False,       # epoch tracking is handled manually
        exist_ok=True,

        epochs=next_target - current_epoch,
        patience=0,         # custom patience tracking across chunks

        imgsz=640,
        batch=6,

        cache='disk',       # RAM cache has calculation bugs; disk is fast enough
        device='cuda',
        workers=4,
        amp=True,

        save=True,
        save_period=5,

        val=True,
        plots=True,

        seed=69,
        deterministic=False,

        **training_config
    )

    gc.collect()
    torch.cuda.empty_cache()

    if results is not None and model.trainer is not None:
        epochs_ran        = model.trainer.epoch + 1
        actual_last_epoch = current_epoch + epochs_ran - 1
        save_chunk_state(actual_last_epoch)
        fix_chunk_artifacts(current_epoch, next_target)

        current_fitness = model.trainer.best_fitness
        if current_fitness > best_fitness:
            save_early_stop_state(0, current_fitness)
            print(f"   Fitness improved: {best_fitness:.4f} → {current_fitness:.4f} — counter reset")
        else:
            new_no_improve = epochs_no_improve + epochs_ran
            save_early_stop_state(new_no_improve, best_fitness)
            print(f"   No improvement ({new_no_improve}/{PATIENCE} patience used)")

        actual_total = actual_last_epoch + 1

    print(f"\n{'='*60}")
    if next_target >= TOTAL_EPOCHS:
        print(f"TRAINING FULLY COMPLETED! ({TOTAL_EPOCHS}/{TOTAL_EPOCHS} epochs)")
    else:
        print(f"   CHUNK COMPLETED! ({actual_total}/{TOTAL_EPOCHS} epochs)")
    print(f"   Best:  runs/detect/{run_name}/weights/best.pt")
    print(f"   Last:  runs/detect/{run_name}/weights/last.pt")
    print(f"{'='*60}\n")

except KeyboardInterrupt:
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n{'='*60}")
    print(f"   INTERRUPTED — progress saved")
    print(f"{'='*60}\n")

except SystemExit:
    # Let the early-stop guard propagate cleanly
    raise

except Exception as e:
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n{'='*60}")
    print(f"ERROR: {str(e)}")
    print(f"{'='*60}\n")
    raise